# Credit-Risk Latent Factor Modeling with PROC HPPLS

## Executive Summary

A retail bank wants to predict three correlated credit-risk outcomes — card utilization, debt-to-income trajectory, and a default-probability proxy — from a wide set of highly collinear bureau-style and macroeconomic predictors. Ordinary regression breaks down under this collinearity, so this notebook uses **PROC HPPLS** (high-performance partial least squares) to extract a handful of latent factors that jointly explain the predictors and all three responses, then validates the factor count with a van der Voet cross-validation test on a held-out portfolio segment.

## Data Sources

All data is generated synthetically inside the notebook with `call streaminit(20240531)` — no external files or network access.

| Dataset | Rows | Variable | Role | Description |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | ID | Unique customer key carried to the scored output |
| | | `Segment` | CLASS predictor | Portfolio segment: Retail, Affluent, Business |
| | | `b1`–`b12` | Predictors | 12 collinear monthly bureau-style behavioral signals |
| | | `RatePctChg` | Predictor | Customer-level interest-rate change exposure |
| | | `InquiryCount` | Predictor | Count of recent hard credit inquiries |
| | | `Utilization` | Response | Revolving credit utilization (%) |
| | | `DTIChange` | Response | Change in debt-to-income ratio |
| | | `DefaultProp` | Response | Default-probability proxy (0–1) |
| | | `Role` | Partition | TRAIN (~75%) / TEST (~25%) validation flag |

# Credit-Risk Latent Factor Modeling with PROC HPPLS

Banks routinely face the **wide, collinear predictor** problem: dozens of monthly bureau attributes, macroeconomic exposures, and behavioral signals that move together, used to predict several risk outcomes that are themselves correlated. Ordinary least squares is unstable here because the predictor matrix is near-singular.

**Partial least squares (PLS)** solves this by extracting a small number of latent factors from the *cross-covariance* of the predictors (X) and the responses (Y), so the factors are tuned to predict the outcomes — not just to summarize X. `PROC HPPLS` is the high-performance counterpart to `PROC PLS`, adding multithreaded execution, data partitioning for validation, and the van der Voet randomization test for choosing the number of factors.

In this notebook we build a single PLS model that simultaneously predicts three correlated credit-risk responses and use a held-out validation segment to confirm how many latent factors the data actually support.

## Step 1 — Generate a synthetic credit-risk panel

We simulate 600 customers. Two unobserved latent drivers (`stress` and `tenure`) generate twelve collinear bureau signals `b1`–`b12` plus rate and inquiry exposures — exactly the high-correlation structure PLS is designed for. The three responses (`Utilization`, `DTIChange`, `DefaultProp`) are different linear combinations of the same drivers, so they too are correlated. A `Role` flag holds out roughly a quarter of the book as a validation segment.

In [1]:
data credit;
   call streaminit(20240531);
   length Segment $8 Role $5;
   do CustomerID = 1 to 600;
      /* customer segment (categorical predictor) */
      u = rand('uniform');
      if u < 0.34 then Segment = 'Retail';
      else if u < 0.70 then Segment = 'Affluent';
      else Segment = 'Business';

      /* unobserved macro / behavioral drivers (collinear) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12 collinear monthly bureau-style predictors b1-b12 */
      array b{12} b1-b12;
      do j = 1 to 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      end;

      /* macro covariates, also tied to the drivers */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( max(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* three correlated credit-risk responses */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      if DefaultProp < 0 then DefaultProp = 0;

      /* hold out ~25% as a validation segment */
      if rand('uniform') < 0.25 then Role = 'TEST';
      else Role = 'TRAIN';

      output;
   end;
   drop u stress tenure j;
run;

NOTE: DATA credit


NOTE: Wrote credit (600 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.03 seconds
  cpu   0.03 seconds


## Step 2 — Fit the multi-response PLS model with cross-validation

The core call demonstrates the principal `PROC HPPLS` statements and the options a risk modeler would actually reach for:

- **`MODEL`** lists all three responses on the left and the full collinear predictor set on the right; **`/ solution`** prints the final predictive coefficients on the raw scale.
- **`CLASS Segment`** brings the portfolio segment in as a categorical predictor (it must precede `MODEL`).
- **`ID CustomerID`** carries the customer key into the scored output.
- **`CVTEST(stat=press ...)`** runs the van der Voet randomization test to pick the number of factors objectively rather than by eye; `seed=` makes it reproducible.
- **`PARTITION rolevar=Role(...)`** fits and scores on the training rows and holds out the `TEST` segment so the cross-validation PRESS is measured out-of-sample.
- **`VARSS`** and **`DETAILS`** report how much X- and Y-variation each successive factor explains.
- **`OUTPUT`** writes predicted values, residuals, X-scores, and PRESS for the fitted (training) rows to a scored dataset, keyed by `CustomerID`.

In [2]:
proc hppls data=credit nfac=8
           cvtest(stat=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   class Segment;
   id CustomerID;
   model Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / solution;
   partition rolevar=Role(train='TRAIN' test='TEST');
   output out=scored predicted=Pred residual=Resid
          xscore=Factor press=Press role=AssignedRole;
run;


The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 600
Number of Observations Used: 441
Number of Training Observations: 441
Number of Test Observations:     159

Class Level Information

  Class Segment: 3 levels: Affluent Business Retail

Response Variable(s): Utilization DTIChange DefaultProp
Predictor Variable(s): b1 b2 b3 b4 b5 b6 b7 b8 b9 b10 b11 b12 RatePctChg InquiryCount Segment

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.6377 74.6377     28.7618 28.7618
    2      4.2004 78.8381     45.0441 73.8059
    3      8.6058 87.4438      1.0025 74.8084
    4      1.8604 89.3042      0.7591 75.5675
    5      2.6230 91.9273      0.1670 75.7345
    6      0.8001 92.7274      0.2035 75.9380
    7      0.6769 93.4043      0.0634 76.0013
    8      0.7059 94.1102      0.0148 76.0161

Variation Accounted for by Var

NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## Step 3 — Summarize the predicted risk profile

With the model fit, we profile the predicted outcomes across the book. `PROC MEANS` reports the central tendency and spread of each predicted response so the risk team can sanity-check the scale (e.g., predicted utilization centered in the mid-40s, default proxy near the assumed 8% base rate).

In [3]:
proc means data=scored mean std min max maxdec=3;
   var Pred_Utilization Pred_DTIChange Pred_DefaultProp;
run;

                                                  The MEANS Procedure

 Variable                   Mean     Std Dev     Minimum     Maximum
 -------------------------------------------------------------------
 PRED_UTILIZATION         44.856      11.923      12.382      83.640
 PRED_DTICHANGE           -0.019       2.511      -6.363       8.428
 PRED_DEFAULTPROP          0.080       0.047      -0.035       0.240
 -------------------------------------------------------------------



NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Step 4 — Inspect individual scored customers

Finally we list a few customers from the fitted **training** segment with their original `Role` flag, predicted utilization, and residual. (The held-out `TEST` rows are deliberately not scored, so we filter to `Role='TRAIN'` to show fully populated rows.) This is the kind of row-level output an analyst would attach to a model-monitoring report or feed downstream to a limit-setting engine.

In [4]:
proc print data=scored(obs=8);
   where Role = 'TRAIN';
   var CustomerID Role Pred_Utilization Resid_Utilization;
run;


  Obs  CUSTOMERID   ROLE  PRED_UTILIZATION  RESID_UTILIZATION
-----  ----------  -----  ----------------  -----------------
    1           2  TRAIN     40.0855648209       9.8577051885
    2           3  TRAIN     38.7582754498       1.7476030913
    3           5  TRAIN     37.0565525834      -5.4030465401
    4           6  TRAIN     61.1069095355       0.0345102526
    5           7  TRAIN     43.0479880937      -2.9054202281
    6          10  TRAIN     50.4479049267      11.3560248797
    7          11  TRAIN     53.2417742625       1.5888775472
    8          12  TRAIN     51.2662201913       4.1320269051

... more observations (showing 8)



NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 8 observations printed, 4 variables


## Interpreting the results

The **Percent Variation Accounted for** table shows the first factor alone absorbs roughly three-quarters of the predictor variation (74.64%, the dominant collinear "stress" direction) but only 28.76% of the *response* variation. The second factor is where the bulk of the response signal is captured: it adds just 4.20% more X-variation yet 45.04% more Y-variation, lifting the cumulative response variation to 73.81%. This is the hallmark of PLS reorienting components toward prediction rather than pure X-variance — the most predictive direction is the *second* factor, not the most X-dominant one. By eight factors the per-response R-squares settle at 0.79 (Utilization), 0.76 (DTIChange), and 0.74 (DefaultProp), confirming the three credit-risk outcomes are well captured by a low-dimensional latent structure.

The **van der Voet PRESS cross-validation** test is the decision-maker here: PRESS on the held-out segment drops sharply through the first three factors (56543 → 20938 → 19513 → 19046), then improves only marginally and bottoms out at factor 6 (18993) before ticking back up at factors 7–8. The test therefore selects **six factors** as the model with the best out-of-sample predictive accuracy. Note the tension a modeler must weigh: factors 4–6 each add well under 1% of cumulative response variation, yet they still shave the held-out PRESS — extracting beyond factor 6 chases noise in the wide bureau block and degrades out-of-sample performance, precisely the overfitting a credit-risk model must avoid before deployment.

The `SOLUTION` coefficients give the bank an interpretable, regularized linear scorecard on the original variable scale, with `RatePctChg` (0.90, 0.88, 0.83 across the three outcomes) and `InquiryCount` (0.41, 0.46, 0.49) emerging as the strongest single drivers — both materially larger than any individual bureau signal `b1`–`b12`, whose collinearity PLS has folded into the shared factors. In practice a modeler would now refit with `nfac=6`, monitor the residuals in the `scored` dataset, and promote the factor scores or coefficients into a production risk-decisioning pipeline.